In [ ]:
from dataclasses import replace

import jax
import jax.numpy as jnp
import jax.scipy as jsp
import matplotlib as mplib
import matplotlib.pyplot as plt
import numpy as np
import optax
import shapely.geometry as shap_geom
from flax import nnx
from shapely.prepared import prep

import moto.src.bc as _bc
import moto.src.cons_loss as _cons_loss
import moto.src.ext_force as _ext_force
import moto.src.geometry as _geom
import moto.src.material as _mat
import moto.src.material_points as _mp
import moto.src.mesher as _mesher
import moto.src.mpm_elem_map as _mpm_elem_map
import moto.src.network as _net
import moto.src.nl_solver as _nlsolv
import moto.src.solve as _solve
import moto.src.utils as _utils
import moto.src.viz as _viz
from moto.src.hyperelastic_mpm import HyperelasticMPM, StructField

_Ext = _utils.Extent

jax.config.update("jax_enable_x64", True)
GIF_DIR = "frames"
GIF_PATH = "mpm.gif"
GIF_FPS = 3
frames = []

_Field = StructField

## Geometry & Mesh

In [ ]:
bbox = _geom.BrepGeometry("moto/brep/rect_domain_expt_5.json")
gripper_bbox = _geom.BrepGeometry("moto/brep/gripper_expt_5.json")
mesh = _mesher.grid_mesh_brep(
  brep=bbox,
  nelx_desired=80,
  nely_desired=40,
  dofs_per_node=2,
  gauss_order=2,
)

## Material Points Generation 


In [ ]:
prepared_beam_geom = prep(gripper_bbox.geometry)
inside = np.array(
  [prepared_beam_geom.covers(shap_geom.Point(xy)) for xy in mesh.elem_centers],
  dtype=bool,
)

# Element IDs to populate
occupied_element_ids = np.where(inside)[0]

# MPs
num_mp_per_elem_per_dim = 4
mp_coords = _mesher.generate_mp_coords_in_occupied_elements(
  mesh,
  occupied_element_ids,
  num_mp_per_elem_per_dim,
)
num_mat_pts = mp_coords.shape[0]

In [ ]:
# ============================================
# EXTRACT VERTICES FROM BREP DIRECTLY
geom = gripper_bbox.geometry
if hasattr(geom, "exterior"):
  coords_list = list(geom.exterior.coords)
else:
  geoms = [g for g in getattr(geom, "geoms", []) if hasattr(g, "exterior")]
  coords_list = list(geoms[0].exterior.coords)

vertices = np.array(coords_list[:-1])
print("BREP Vertices:")
for i, v in enumerate(vertices):
  print(f"  V{i}: ({v[0]:.4f}, {v[1]:.4f})")

# ============================================
# IDENTIFY KEY VERTICES BY GEOMETRY
# Input: highest y (top of vertical edge)
input_idx = np.argmax(vertices[:, 1])
input_vertex = vertices[input_idx]

output_idx = np.argmin(vertices[:, 1])
output_vertex = vertices[output_idx]

# Clamp: rightmost x, lower portion (junction area)
# Find vertex with max x but not the input
right_vertices = vertices[vertices[:, 0] == vertices[:, 0].max()]
clamp_vertex = right_vertices[np.argmin(right_vertices[:, 1])]

print(f"\nKey positions:")
print(f"  Input (top):    ({input_vertex[0]:.4f}, {input_vertex[1]:.4f})")
print(f"  Output (bottom): ({output_vertex[0]:.4f}, {output_vertex[1]:.4f})")
print(f"  Clamp (right-low): ({clamp_vertex[0]:.4f}, {clamp_vertex[1]:.4f})")

# ============================================
# COMPUTE EDGE DIRECTIONS
edge_start = output_vertex
edge_end = vertices[(output_idx + 2) % len(vertices)]  # Next vertex

edge_vec = edge_end - edge_start
edge_length = np.linalg.norm(edge_vec)
edge_dir = edge_vec / edge_length

# Perpendicular to edge
perp_dir = np.array([edge_dir[1], -edge_dir[0]])

# Make sure perpendicular points away from gripper centroid
centroid = np.mean(vertices, axis=0)
test_point = edge_start + 0.01 * perp_dir
if np.linalg.norm(test_point - centroid) < np.linalg.norm(edge_start - centroid):
  perp_dir = -perp_dir  # Flip if pointing toward centroid

print(f"\n45° Edge direction: ({edge_dir[0]:.3f}, {edge_dir[1]:.3f})")
print(f"Perpendicular (outward): ({perp_dir[0]:.3f}, {perp_dir[1]:.3f})")


# Non design Solid region

In [ ]:
hx, hy = map(float, np.asarray(mesh.elem_size))
face_tol = 0.5 * max(hx, hy)
xminG, yminG, xmaxG, ymaxG = gripper_bbox.geometry.bounds
nd_width_in = 0.005
nd_width_out = 0.002
# Non-design: from start to past the clamp end
t_start = 0.0
t_end = 1.05

p1 = edge_start + t_start * edge_vec
p2 = edge_start + t_end * edge_vec

# Extend in BOTH directions from the edge
p1_out = p1 + nd_width_out * perp_dir
p2_out = p2 + nd_width_out * perp_dir

p1_in = p1 - nd_width_in * perp_dir
p2_in = p2 - nd_width_in * perp_dir

nd_polygon = shap_geom.Polygon([p1_out, p2_out, p2_in, p1_in, p1_out])

# Find MPs in non-design region
solid_id = 0
mp_xy = np.asarray(mp_coords)
mask_nd = np.array([nd_polygon.contains(shap_geom.Point(xy)) for xy in mp_xy])
nd_ids_np = np.where(mask_nd)[0].astype(np.int32)
nd_ids = jnp.asarray(nd_ids_np)
print(f"Non-design MPs: {nd_ids.size}")


# Boundary conditions

In [ ]:
# --------------------------
# CLAMP: At clamp_vertex
clamp_x, clamp_y = clamp_vertex

clamp_bbox = _mesher.BoundingBox(
  x=_Ext(min=clamp_x - 2 * face_tol, max=clamp_x + 3 * face_tol),
  y=_Ext(min=clamp_y - 2 * face_tol, max=clamp_y + 3 * face_tol),
)
mask_clamp_nodes = _mesher.compute_point_indices_in_box(
  np.asarray(mesh.nodes.coords), clamp_bbox
)
clamp_nodes = np.where(np.asarray(mask_clamp_nodes))[0]
if clamp_nodes.size == 0:
  raise ValueError("No nodes in clamp bbox")

fixed_dofs_clamp = np.r_[2 * clamp_nodes, 2 * clamp_nodes + 1].astype(np.int32)
print(f"Clamp nodes: {clamp_nodes.size} at ({clamp_x:.4f}, {clamp_y:.4f})")

# --------------------------
# ROLLER at INPUT
input_x, input_y = input_vertex

roller_input_bbox = _mesher.BoundingBox(
  x=_Ext(min=input_x - 2 * face_tol, max=input_x + 2 * face_tol),
  y=_Ext(min=input_y - 2 * face_tol, max=input_y + 2 * face_tol),
)
mask_roller_input = _mesher.compute_point_indices_in_box(
  np.asarray(mesh.nodes.coords), roller_input_bbox
)
roller_input_nodes = np.where(np.asarray(mask_roller_input))[0]
if roller_input_nodes.size == 0:
  raise ValueError("No nodes in input roller bbox")

fixed_dofs_roller_input = (2 * roller_input_nodes).astype(np.int32)  # X only
print(
  f"Input roller nodes: {roller_input_nodes.size} at ({input_x:.4f}, {input_y:.4f})"
)

# --------------------------
# Combine all fixed DOFs
fixed_dofs = np.unique(
  np.r_[
    fixed_dofs_clamp,
    fixed_dofs_roller_input,
  ]
).astype(np.int32)

all_dofs = np.arange(mesh.num_dofs, dtype=np.int32)
free_dofs = np.setdiff1d(all_dofs, fixed_dofs).astype(np.int32)

bc = {
  "fixed_dofs": jnp.asarray(fixed_dofs),
  "free_dofs": jnp.asarray(free_dofs),
}
print(f"Total fixed DOFs: {fixed_dofs.size}")


#  Forces on material points

In [ ]:
x_in, y_in = input_vertex

in_bbox = _mesher.BoundingBox(
  x=_Ext(min=x_in - 2 * face_tol, max=x_in + 2 * face_tol),
  y=_Ext(min=y_in - 2 * face_tol, max=y_in + 2 * face_tol),
)
mask_in = _mesher.compute_point_indices_in_box(np.asarray(mp_coords), in_bbox)
in_ids = np.where(np.asarray(mask_in))[0]
if in_ids.size == 0:
  raise ValueError("No MPs found in input bbox")

F_in_total = -10.0  # Negative y = downward

mp_force_in = jnp.zeros((num_mat_pts, mesh.num_dim), dtype=jnp.float64)
mp_force_in = mp_force_in.at[in_ids, 1].set(F_in_total / in_ids.size)

print(f"\nInput force at ({x_in:.4f}, {y_in:.4f})")
print(f"  MPs: {in_ids.size}, Force: (0, {F_in_total}) - DOWN")


# Virtual Force

In [ ]:
x_out, y_out = output_vertex

out_bbox = _mesher.BoundingBox(
  x=_Ext(min=x_out - 2 * face_tol, max=x_out + 2 * face_tol),
  y=_Ext(min=y_out - 3 * face_tol, max=y_out + 2 * face_tol),
)
mask_out = _mesher.compute_point_indices_in_box(np.asarray(mp_coords), out_bbox)
out_ids = np.where(np.asarray(mask_out))[0]
if out_ids.size == 0:
  raise ValueError("No MPs found in output bbox")

out_ids = jnp.asarray(out_ids)

# Force PERPENDICULAR to edge, pointing toward workpiece (outward)
F_out_magnitude = 10.0
F_out_x = F_out_magnitude * perp_dir[0]
F_out_y = F_out_magnitude * perp_dir[1]

mp_force_out = jnp.zeros((num_mat_pts, mesh.num_dim), dtype=jnp.float64)
mp_force_out = mp_force_out.at[out_ids, 0].set(F_out_x / out_ids.size)
mp_force_out = mp_force_out.at[out_ids, 1].set(F_out_y / out_ids.size)

print(f"\nOutput force at ({x_out:.4f}, {y_out:.4f})")
print(f"  MPs: {out_ids.size}")
print(f"  Force direction: ({F_out_x:.3f}, {F_out_y:.3f}) - perpendicular to edge")


## Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

Xn = np.asarray(mesh.nodes.coords)
mp_xy = np.asarray(mp_coords)

# All grid nodes
ax.scatter(Xn[:, 0], Xn[:, 1], s=1, c="lightgray", alpha=0.3, label="Grid nodes")

# Gripper polygon (orange)
xs_grip, ys_grip = (
  geom.exterior.xy if hasattr(geom, "exterior") else geoms[0].exterior.xy
)
ax.plot(xs_grip, ys_grip, "orange", lw=2, label="Gripper domain")

# Non-design region (blue)
if nd_ids.size > 0:
  xs_nd, ys_nd = nd_polygon.exterior.xy
  ax.fill(xs_nd, ys_nd, alpha=0.3, color="blue", label="Non-design")
  ax.scatter(mp_xy[nd_ids, 0], mp_xy[nd_ids, 1], s=3, c="blue", alpha=0.7)

# Clamp (black squares)
ax.scatter(
  Xn[clamp_nodes, 0],
  Xn[clamp_nodes, 1],
  s=100,
  c="black",
  marker="s",
  label="Clamp (fix x,y)",
  zorder=10,
)

# Input roller (purple triangles)
ax.scatter(
  Xn[roller_input_nodes, 0],
  Xn[roller_input_nodes, 1],
  s=100,
  c="purple",
  marker=">",
  label="Input roller (fix x)",
  zorder=10,
)

# Input force (red) - pointing DOWN
ax.scatter(mp_xy[in_ids, 0], mp_xy[in_ids, 1], s=40, c="red", zorder=5)
arrow_len = 0.02
ax.annotate(
  "",
  xy=(x_in, y_in - arrow_len),
  xytext=(x_in, y_in),
  arrowprops=dict(arrowstyle="->", color="red", lw=3),
)
ax.text(x_in + 0.008, y_in - arrow_len / 2, "$F_{in}$ (DOWN)", fontsize=11, color="red")

# Output force (green) - pointing PERPENDICULAR to edge
ax.scatter(mp_xy[out_ids, 0], mp_xy[out_ids, 1], s=40, c="green", zorder=5)
ax.annotate(
  "",
  xy=(x_out + arrow_len * perp_dir[0], y_out + arrow_len * perp_dir[1]),
  xytext=(x_out, y_out),
  arrowprops=dict(arrowstyle="->", color="green", lw=3),
)
ax.text(
  x_out + arrow_len * perp_dir[0],
  y_out + arrow_len * perp_dir[1] - 0.01,
  "$F_{out}$ (⊥ edge)",
  fontsize=11,
  color="green",
)

# Mark all BREP vertices
for i, v in enumerate(vertices):
  ax.plot(v[0], v[1], "ko", markersize=8)
  ax.text(v[0] + 0.003, v[1] + 0.003, f"V{i}", fontsize=9, color="black")

# Draw the 45° edge direction
mid_edge = 0.5 * (edge_start + edge_end)
ax.annotate(
  "",
  xy=mid_edge + 0.02 * edge_dir,
  xytext=mid_edge,
  arrowprops=dict(arrowstyle="->", color="gray", lw=2),
)
ax.text(mid_edge[0], mid_edge[1] + 0.01, "edge dir", fontsize=9, color="gray")


# Rect bbox
def plot_bbox_fn(ax, bb, label):
  xmin, ymin, xmax, ymax = bb.geometry.bounds
  ax.plot(
    [xmin, xmax, xmax, xmin, xmin], [ymin, ymin, ymax, ymax, ymin], lw=2, label=label
  )


plot_bbox_fn(ax, bbox, "Rect bbox")

ax.set_aspect("equal")
ax.legend(loc="upper left", fontsize=9)
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_title(
  "Rotated Gripper Setup\n"
  f"Input: DOWN at V{input_idx} | Output: ⊥ to edge at V{output_idx} | Clamp at right"
)
plt.grid(True, alpha=0.3)
plt.show()

## Material Initialization

In [ ]:
youngs_modulii = jnp.array([12.0, 3.0, 1.0, 1e-4]) * 1e9
mass_densities = jnp.array([4.0, 1.6, 0.75, 1e-4]) * 1e3

poissons_ratios = jnp.array([0.4, 0.4, 0.4, 0.4])
yield_strengths = jnp.array([2.0, 1.2, 0.8, 1e-4]) * 1e5

lame_lam, lame_mu = _mat.get_lame_parameters_from_youngs_modulus_and_poissons_ratio(
  youngs_modulii, poissons_ratios
)

num_materials = len(youngs_modulii)
cmap = mplib.colors.ListedColormap(_viz.mat_colors[:num_materials])

In [ ]:
# MP half-lengths and volumes
thickness = 1.0e-3
elem_size = jnp.asarray(mesh.elem_size)
half_length_per_mp = elem_size / (2 * num_mp_per_elem_per_dim)
volume_per_mp = (
  thickness * jnp.prod(elem_size) / (num_mp_per_elem_per_dim**mesh.num_dim)
)

max_nodes = 3**mesh.num_dim
max_elems = 2**mesh.num_dim

mp_state, _ = _mp.initialize_new_material_points(
  num_pts=num_mat_pts,
  num_dim=mesh.num_dim,
  max_nodes_per_point=max_nodes,
  max_elems_per_point=max_elems,
)
mass_per_mp = volume_per_mp * jnp.mean(mass_densities)  # Assign avg density initially

mp_state = replace(
  mp_state,
  coord=mp_coords,
  volume=volume_per_mp * jnp.ones((num_mat_pts,), dtype=jnp.float64),
  volume0=volume_per_mp * jnp.ones((num_mat_pts,), dtype=jnp.float64),
  mass=mass_per_mp * jnp.ones((num_mat_pts,), dtype=jnp.float64),
  domain_length=jnp.tile(half_length_per_mp[None, :], (num_mat_pts, 1)),
  domain_length0=jnp.tile(half_length_per_mp[None, :], (num_mat_pts, 1)),
)

## Density filter

In [ ]:
density_filter = _utils.create_density_filter(
  coords=mp_coords,
  cutoff_distance=4.0 * mesh.elem_size[0],
  filter_type=_utils.Filters.CIRCULAR,
)

## Network and projections

In [ ]:
symm_map = _net.Symmetry()
fourier_proj = _net.FourierProjection(
  num_input_dim=2,
  num_terms=100,
  min_radius=0.008,
  max_radius=0.5,
)

topnet = _net.TopNet(
  num_neurons=[2 * fourier_proj.num_terms, 40, 40, num_materials],
  rngs=nnx.Rngs(1234),
  use_batch_norm=True,
)

In [ ]:
pts_xy = symm_map.apply(mp_state.coord)
pts_xy_fourier = fourier_proj.apply(pts_xy)

In [ ]:
loss_params = [_cons_loss.LogBarrierParams(t0=3.0, mu=1.02)]

## Solver Setup

In [ ]:
nr_tol, max_nr_iter = 1e-7, 60

solver_settings = {
  "linear": {"solver": _nlsolv.LinearSolvers.SCIPY_SPARSE, "rtol": 1e-12},
  "nonlinear": {
    "max_iter": max_nr_iter,
    "threshold": nr_tol,
    "lam_min": 0.01,
    "line_search_max_iter": 12,
    "line_search_shrink": 0.5,
    "line_search_alpha_min": 1e-6,
    "line_search_armijo_c": 1e-4,
  },
}

mpm_problem = HyperelasticMPM(
  solver_settings=solver_settings,
  mesh=mesh,
)

du_guess = jnp.zeros((mesh.num_dofs,)) + 1e-3
du_guess = du_guess.at[bc["fixed_dofs"]].set(0.0)
gravity_vec = jnp.zeros((mesh.num_dim,))

In [ ]:
eta = 0.5
rho_min = 1e-3

In [ ]:
@nnx.jit(static_argnames=("num_load_steps",))
def loss_function(
  net: _net.TopNet,
  mp_state0: _mp.MaterialPointConfig,
  penal: float,
  num_load_steps: int,
  gravity_vec: jnp.ndarray,
  max_mass: float,
  obj_0: float,
  epoch: int,
) -> jnp.ndarray:
  # compute pseudo-densities
  pseudo_densities = jax.nn.softmax(net(pts_xy_fourier), axis=-1)
  solid_vec = jax.nn.one_hot(
    solid_id, pseudo_densities.shape[-1], dtype=pseudo_densities.dtype
  )  # (M,)
  pseudo_densities = pseudo_densities.at[nd_ids].set(solid_vec)
  penal_matfrac = rho_min + (1.0 - rho_min) * (pseudo_densities**penal)

  lam = jnp.einsum("m, pm -> p", lame_lam, penal_matfrac)
  mu = jnp.einsum("m, pm -> p", lame_mu, penal_matfrac)

  matpt_massdens = jnp.einsum("m, pm -> p", mass_densities, pseudo_densities)
  matpt_mass = matpt_massdens * mp_state0.volume0

  mp_state_in = replace(mp_state0, pseudo_density=pseudo_densities, mass=matpt_mass)

  # solve MPM and compute objective
  mp_final_in, _ = _solve.newton_solve(
    mesh=mesh,
    mp_state=mp_state_in,
    bc=bc,
    du_guess=du_guess,
    load_steps=num_load_steps,
    gravity=gravity_vec,
    mpm_problem=mpm_problem,
    lame_lambda=lam,
    lame_mu=mu,
    mp_point_force=mp_force_in,
  )

  mp_final_out, _ = _solve.newton_solve(
    mesh=mesh,
    mp_state=mp_state_in,
    bc=bc,
    du_guess=du_guess,
    load_steps=num_load_steps,
    gravity=gravity_vec,
    mpm_problem=mpm_problem,
    lame_lambda=lam,
    lame_mu=mu,
    mp_point_force=mp_force_out,
  )
  strain_enrgy_in = jnp.einsum("pd, pd -> ", mp_force_in, mp_final_in.displacement)
  strain_enrgy_out = jnp.einsum("pd, pd -> ", mp_force_out, mp_final_out.displacement)
  mutual_strain_energy = jnp.einsum(
    "pd, pd -> ", mp_force_in, mp_final_out.displacement
  )
  obj = -mutual_strain_energy / (strain_enrgy_in + strain_enrgy_out)

  # volume constraint
  net_mass = jnp.sum(matpt_mass)
  mass_cons = (net_mass / max_mass) - 1.0  # <= 0

  # loss
  loss = _cons_loss.combined_loss(
    obj / obj_0,
    jnp.array([mass_cons]),
    [_cons_loss.ConstraintTypes.INEQUALITY],
    loss_params,
    epoch,
  )

  return loss, (mp_final_in, pseudo_densities, obj, net_mass)

## Optimization

In [ ]:
def optimize_design(
  net: _net.TopNet,
  mp_state0: _mp.MaterialPointConfig,
  max_mass: float,
  max_epoch: int,
  num_load_steps: int = 15,
  gravity_vec=None,
  lr: float = 1e-2,
  plot_interval: int = 2,
):
  # gravity vector
  if gravity_vec is None:
    gravity_vec = jnp.zeros((mesh.num_dim,), dtype=jnp.float64)

  obj_0 = 1.0
  opt = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adam(lr),
  )
  optimizer = nnx.ModelAndOptimizer(topnet, opt)

  for epoch in range(max_epoch):
    penal = min(5.0, 1.0 + epoch * 0.05)

    (loss, (mp_final, mat_fracs, obj, vol_frac)), grad_loss = nnx.value_and_grad(
      loss_function, has_aux=True
    )(net, mp_state0, penal, num_load_steps, gravity_vec, max_mass, obj_0, epoch)

    optimizer.update(grad_loss)

    status = f"epoch {epoch} Loss {loss:.2E} J {obj:.2E} vf {vol_frac:.2F}"
    print(status)

    if epoch == 0 or epoch == 10:
      obj_0 = jax.lax.stop_gradient(jnp.abs(obj))

    if epoch % plot_interval == 0:
      coords = np.asarray(mp_final.coord)
      mat_idx = jnp.argmax(mat_fracs, axis=-1)

      _, ax = plt.subplots()
      img = ax.scatter(coords[:, 0], coords[:, 1], s=2, c=mat_idx, cmap=cmap)
      ax.set_xlim([float(mesh.bounding_box.x.min), float(mesh.bounding_box.x.max)])
      ax.set_ylim([float(mesh.bounding_box.y.min), float(mesh.bounding_box.y.max)])
      plt.colorbar(img, ax=ax)
      plt.show()
      plt.pause(1e-6)

  return mp_final, mat_fracs, net


In [ ]:
mass_frac = 0.6
V_domain = jnp.sum(mp_state.volume0)
rho_ref = jnp.mean(mass_densities[:-1])
max_mass = mass_frac * V_domain * rho_ref

In [ ]:
mp_final, mat_fracs, net = optimize_design(
  net=topnet,
  mp_state0=mp_state,
  max_mass=max_mass,
  max_epoch=220,
  lr=1e-2,
  num_load_steps=1,
  gravity_vec=gravity_vec,
)

In [ ]:
plt.rcParams.update(_viz.high_res_plot_settings)
coords = np.asarray(mp_final.coord)
mat_idx = jnp.argmax(mat_fracs, axis=-1)

_, ax = plt.subplots()
img = ax.scatter(coords[:, 0], coords[:, 1], s=2, c=mat_idx, cmap=cmap)
ax.set_xlim([float(mesh.bounding_box.x.min), float(mesh.bounding_box.x.max)])
ax.set_ylim([float(mesh.bounding_box.y.min), float(mesh.bounding_box.y.max)])
ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
ax.set_xticks([])
ax.set_yticks([])

# cbar = plt.colorbar(img, ax=ax, ticks=[0, 1, 2, 3])
# cbar.set_ticklabels(["0", "1", "2", "3"])  # optional, but explicit
ax.set_aspect("equal")

plt.show()


In [ ]:
num_mp_per_elem_per_dim = 4
bbox = _geom.BrepGeometry("moto/brep/rect_domain_expt_5.json")
high_res_mesh = _mesher.grid_mesh_brep(
  brep=bbox,
  nelx_desired=200,
  nely_desired=100,
  dofs_per_node=2,
  gauss_order=2,
)
inside = np.array(
  [prepared_beam_geom.covers(shap_geom.Point(xy)) for xy in high_res_mesh.elem_centers],
  dtype=bool,
)

# Element IDs to populate
occupied_element_ids = np.where(inside)[0]
mp_coords_high_res = _mesher.generate_mp_coords_in_occupied_elements(
  high_res_mesh,
  occupied_element_ids,
  num_mp_per_elem_per_dim,
)
mp_xy_res = np.asarray(mp_coords_high_res)
mask_nd_res = np.array([nd_polygon.contains(shap_geom.Point(xy)) for xy in mp_xy_res])
nd_ids_np_res = np.where(mask_nd_res)[0].astype(np.int32)


if nd_ids_np.size == 0:
  raise ValueError(
    "No MPs in non-design bbox; increase height_non_design or check coordinates."
  )

nd_ids_res = jnp.asarray(nd_ids_np_res)

pts_xy_high_res = symm_map.apply(mp_coords_high_res)


pts_xy_high_res_fourier = fourier_proj.apply(pts_xy_high_res)
mat_fracs_high_res = jax.nn.softmax(net(pts_xy_high_res_fourier), axis=-1)

solid_vec = jax.nn.one_hot(
  solid_id, mat_fracs_high_res.shape[-1], dtype=mat_fracs_high_res.dtype
)  # (M,)
mat_fracs_high_res = mat_fracs_high_res.at[nd_ids_res].set(solid_vec)
mat_idx_high_res = jnp.argmax(mat_fracs_high_res, axis=-1)

_, ax = plt.subplots()
img = ax.scatter(
  mp_coords_high_res[:, 0], mp_coords_high_res[:, 1], s=2, c=mat_idx_high_res, cmap=cmap
)
ax.set_xlim([float(mesh.bounding_box.x.min), float(mesh.bounding_box.x.max)])
ax.set_ylim([float(mesh.bounding_box.y.min), float(mesh.bounding_box.y.max)])
ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
ax.set_xticks([])
ax.set_yticks([])
ax.set_aspect("equal")
plt.show()